<a href="https://colab.research.google.com/github/Innovatewithapple/RNNProjects/blob/main/TwitterSentiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install lime

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 10.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for lime: filename=lime-0.2.0.1-py3-none-any.whl size=283834 sha256=a102f7191ea4b458c91d9336af8400efad4bcaf17ac0318033109320ed46c663
  Stored in directory: /root/.cache/pip/wheels/e7/5d/0e/4b4fff9a47468fed5633211fb3b76d1db43fe806a17fb7486a
Successfully built lime


In [3]:
import tensorflow as tf
from tensorflow.keras.layers import Layer,TextVectorization,Dense,SimpleRNN,LSTM,Bidirectional,Embedding,Dropout,Input
from sklearn.model_selection import train_test_split
from tensorflow.keras import Sequential
from lime.lime_text import LimeTextExplainer
import numpy as np
import os
from google.colab import userdata
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.callbacks import EarlyStopping

In [4]:
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
print('activated')

activated


In [5]:
!kaggle datasets download -d jp797498e/twitter-entity-sentiment-analysis

Dataset URL: https://www.kaggle.com/datasets/jp797498e/twitter-entity-sentiment-analysis
License(s): CC0-1.0
100% 1.99M/1.99M [00:00<00:00, 161MB/s]



In [6]:
!unzip -q twitter-entity-sentiment-analysis.zip -d ./dataset_folder

In [7]:
train_Dataset = pd.read_csv('/content/dataset_folder/twitter_training.csv')
val_Dataset = pd.read_csv('/content/dataset_folder/twitter_validation.csv')

In [8]:
train_Dataset.sample(5)

,2401,Borderlands,Positive,"im getting on borderlands and i will murder you all ,"
15778,3099,Dota2,Negative,use @DOTA2 queue for long -> accept found matc...
50002,6183,FIFA,Negative,BRO IN PHYSICALLY CANNOT PLAY WITH ANY FUCKING...
51665,10474,RedDeadRedemption(RDR),Neutral,"Childhood, eh?. . Zelda Wind Waker. Grand Thef..."
3389,1788,CallOfDutyBlackopsColdWar,Positive,The new cod is..actually really fun .
11041,13098,Xbox(Xseries),Positive,"Remember, it was one of the best games I've ev..."


In [9]:
train_Dataset.columns

Index(['2401', 'Borderlands', 'Positive',
       'im getting on borderlands and i will murder you all ,'],
      dtype='object')

In [10]:
train_Dataset = train_Dataset.rename(columns={'im getting on borderlands and i will murder you all ,': 'text'})

In [11]:
val_Dataset = val_Dataset.rename(columns={'I mentioned on Facebook that I was struggling for motivation to go for a run the other day, which has been translated by Tom’s great auntie as ‘Hayley can’t get out of bed’ and told to his grandma, who now thinks I’m a lazy, terrible person 🤣': 'text','Irrelevant':'Positive'})

In [12]:
train_Dataset.sample(5)

,2401,Borderlands,Positive,text
49302,6064,FIFA,Negative,"I wouldn't even play that formation on FIFA, a..."
51206,6389,FIFA,Negative,"Fifa 20 Online is pretty woeful, get it? Just ..."
28861,565,ApexLegends,Neutral,If we are allowed to cancel the jump when they...
42834,10148,PlayerUnknownsBattlegrounds(PUBG),Negative,The PTA banning of PUBG is another admission o...
28168,448,ApexLegends,Neutral,Just beat titanfall 2 campaign and embrace it....


In [13]:
train_text = train_Dataset['text'].astype(str).values
train_label = train_Dataset['Positive'].astype(str).values
train_text

array(['I am coming to the borders and I will kill you all,',
       'im getting on borderlands and i will kill you all,',
       'im coming on borderlands and i will murder you all,', ...,
       'Just realized the windows partition of my Mac is now 6 years behind on Nvidia drivers and I have no idea how he didn’t notice',
       'Just realized between the windows partition of my Mac is like being 6 years behind on Nvidia drivers and cars I have no fucking idea how I ever didn ’ t notice',
       'Just like the windows partition of my Mac is like 6 years behind on its drivers So you have no idea how I didn’t notice'],
      dtype=object)

In [14]:
val_Dataset.columns

Index(['3364', 'Facebook', 'Positive', 'text'], dtype='object')

In [15]:
val_text = val_Dataset['text'].astype(str).values
val_label = val_Dataset['Positive'].astype(str).values

In [16]:
# 1. Drop any row that has a NaN in 'text' or your label column
train_Dataset = train_Dataset.dropna(subset=['text', 'Positive'])
val_Dataset = val_Dataset.dropna(subset=['text', 'Positive'])

# 2. Double-check: Are there any empty strings?
# Sometimes text isn't NaN, it's just ""
train_Dataset = train_Dataset[train_Dataset['text'].str.strip() != ""]
val_Dataset = val_Dataset[val_Dataset['text'].str.strip() != ""]

# 3. Re-get your values after cleaning
train_text = train_Dataset['text'].astype(str).values
val_text = val_Dataset['text'].astype(str).values

In [17]:
train_label = train_Dataset['Positive'].astype(str).values
val_label = val_Dataset['Positive'].astype(str).values

In [18]:
label_encoder = LabelEncoder()
train_label_encoded = label_encoder.fit_transform(train_label)
val_label_encoded = label_encoder.fit_transform(val_label)

In [19]:
train_ds = tf.data.Dataset.from_tensor_slices((train_text,train_label_encoded)).batch(32)
val_ds = tf.data.Dataset.from_tensor_slices((val_text,val_label_encoded)).batch(32)

In [20]:
#vertorization
max_words = 10000
max_len = 35

vectorize_layer = TextVectorization(max_tokens=max_words,output_mode='int',output_sequence_length=max_len)

In [21]:
vectorize_layer.adapt(train_text) # adapt basically gives IDs to words

In [27]:
#LSTM
model = Sequential([
    Input(shape=(1,),dtype=tf.string),
    vectorize_layer,
    Embedding(input_dim=max_words,output_dim=128),

    LSTM(64,dropout=0.2,recurrent_dropout=0,return_sequences=True),
    LSTM(64,dropout=0.2,recurrent_dropout=0,return_sequences=False),

    Dense(128,activation='relu'),
    Dropout(0.2),
    Dense(4,activation='softmax')
])

In [28]:
early_stop = EarlyStopping(
    monitor='val_loss',     # 1. Watch the validation loss
    patience=3,              # 2. Wait 3 epochs before giving up
    restore_best_weights=True, # 3. Keep the version that was the 'smartest'
    verbose=1                # 4. Tell us when it stops!
)

In [29]:
print(f"Unique labels: {np.unique(train_label_encoded)}")

Unique labels: [0 1 2 3]


In [30]:
print(label_encoder.classes_)

['Irrelevant' 'Negative' 'Neutral' 'Positive']


In [31]:
model.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])

model.fit(train_ds,validation_data=val_ds,epochs=10,verbose=1,callbacks=[early_stop])

Epoch 1/10
2307/2307 ━━━━━━━━━━━━━━━━━━━━ 26s 10ms/step - accuracy: 0.3628 - loss: 1.3237 - val_accuracy: 0.2913 - val_loss: 1.5201
Epoch 2/10
2307/2307 ━━━━━━━━━━━━━━━━━━━━ 22s 9ms/step - accuracy: 0.4199 - loss: 1.2568 - val_accuracy: 0.4635 - val_loss: 1.3098
Epoch 3/10
2307/2307 ━━━━━━━━━━━━━━━━━━━━ 23s 10ms/step - accuracy: 0.5424 - loss: 1.0917 - val_accuracy: 0.6266 - val_loss: 1.0270
Epoch 4/10
2307/2307 ━━━━━━━━━━━━━━━━━━━━ 41s 10ms/step - accuracy: 0.6415 - loss: 0.9018 - val_accuracy: 0.6847 - val_loss: 0.8781
Epoch 5/10
2307/2307 ━━━━━━━━━━━━━━━━━━━━ 23s 10ms/step - accuracy: 0.7222 - loss: 0.7229 - val_accuracy: 0.7467 - val_loss: 0.7618
Epoch 6/10
2307/2307 ━━━━━━━━━━━━━━━━━━━━ 22s 9ms/step - accuracy: 0.7881 - loss: 0.5770 - val_accuracy: 0.7918 - val_loss: 0.6638
Epoch 7/10
2307/2307 ━━━━━━━━━━━━━━━━━━━━ 23s 10ms/step - accuracy: 0.8311 - loss: 0.4681 - val_accuracy: 0.8338 - val_loss: 0.5648
Epoch 8/10
2307/2307 ━━━━━━━━━━━━━━━━━━━━ 42s 10ms/step - accuracy: 0.8590 - l